# Chapitre 0.1 : Python pour l’IA

Toutes les IA de ce site s’écrivent en Python. Dans ce carnet, tu en apprends juste ce qu’il faut, en faisant tourner un vrai petit programme : il lit un texte, compte quel mot suit quel autre, et devine le mot suivant.

Ce carnet accompagne [la page du chapitre 0.1](https://chipat-neko.github.io/forge-ia/partie-0/0-1/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. manipuler des nombres et du texte, et voir pourquoi `0.1 + 0.2` ne donne pas tout à fait `0.3` ;
2. ranger des mots dans une liste, et en prendre des tranches ;
3. associer chaque mot à son voisin avec `zip`, puis compter avec un dictionnaire et un `Counter` ;
4. faire tourner les fonctions du projet sur un vrai texte ;
5. écrire une suite de 12 mots, et voir pourquoi elle finit par tourner en rond ;
6. relever un défi, puis vérifier que ce carnet trouve exactement les mêmes résultats que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet n’utilise que ce qui est livré avec Python : il n’y a rien d’autre à installer.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "0.1-python"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))
print("Le code du projet est prêt.")

## Étape 1 : des nombres et du texte

Une **variable** est un nom posé sur une valeur, avec `=`. Chaque valeur a un **type** :

- `int`, un nombre entier ;
- `float`, un nombre à virgule, qu’on écrit avec un point : `2.5` ;
- `str`, du texte, entre guillemets.

Une **f-string** commence par `f` avant les guillemets : chaque morceau entre accolades y est remplacé par sa valeur. La fonction `len` donne la longueur d’un texte, en caractères.

In [ ]:
coups = 12        # un entier : int
poids = 2.5       # un nombre à virgule : float (avec un point, pas une virgule)
mot = "forgeron"  # du texte : str

print(type(coups), type(poids), type(mot))
print(f"Le mot {mot} a {len(mot)} lettres, et le forgeron frappe {coups} coups.")

Un nombre à virgule est rangé en binaire, avec un nombre fixe de chiffres. La plupart des nombres décimaux, comme `0.1`, n’y tombent pas juste : ils sont arrondis au plus proche. D’habitude, l’arrondi ne se voit pas. Mais il arrive qu’il se voie, et alors une égalité qui semble évidente devient fausse.

Lance la cellule suivante et regarde les deux dernières lignes. Le module `math` fournit `isclose`, qui compare deux nombres à virgule avec une petite tolérance : c’est ainsi qu’on les compare.

In [ ]:
import math

somme = 0.1 + 0.2
print(somme)
print(somme == 0.3)               # l’égalité exacte échoue
print(math.isclose(somme, 0.3))   # « presque égal », la bonne façon de comparer

## Étape 2 : des listes de mots

Une **liste** range des valeurs dans l’ordre, entre crochets. On lit une case avec son **indice**, qui commence à 0 : `mots[0]` est le premier mot, et `mots[-1]` le dernier.

Une **tranche** prend un morceau de la liste : `mots[1:3]` va de l’indice 1 inclus à l’indice 3 exclu. Sans le second nombre, `mots[1:]`, elle va jusqu’au bout. Une tranche fabrique une nouvelle liste : l’ancienne ne change pas.

In [ ]:
mots = ["le", "forgeron", "frappe", "le", "fer"]

print(mots[0])     # le premier mot
print(mots[-1])    # le dernier
print(mots[1:3])   # une tranche : les indices 1 et 2
print(mots[1:])    # tout sauf le premier
print(f"La liste n’a pas changé, elle a toujours {len(mots)} mots.")
print(mots)

## Étape 3 : compter avec un dictionnaire

Pour savoir quel mot suit quel autre, il faut regarder chaque mot avec son voisin de droite. `zip` avance dans deux listes en même temps : avec `mots` et `mots[1:]`, la même liste décalée d’un cran, il forme toutes les **paires de voisins**.

Un **dictionnaire** associe une clé à une valeur, comme un mot à un nombre. Ici, pour chaque mot qui suit « le », on ajoute 1 à son compte. Le `Counter` du module `collections` est un dictionnaire fait pour compter : un mot qu’il n’a jamais vu vaut 0, et `most_common` donne les plus fréquents.

In [ ]:
from collections import Counter

paires = list(zip(mots, mots[1:]))
print(paires)

# Un dictionnaire : chaque mot qui suit « le », avec son nombre d’apparitions.
apres_le = {}
for mot, suivant in paires:
    if mot == "le":
        apres_le[suivant] = apres_le.get(suivant, 0) + 1
print(apres_le)

# Un Counter fait le même travail sur toute la liste, en une ligne.
compte = Counter(mots)
print(compte)
print(compte.most_common(1))   # le plus fréquent, et son nombre d’apparitions

## Étape 4 : le programme du projet, sur un vrai texte

Le projet range ces idées dans des **fonctions**, dans le fichier `mot_suivant/mots.py` :

- `decouper_mots` met le texte en minuscules et le découpe en mots. Le point, le point d’exclamation et le point d’interrogation y sont des mots à part : ils marquent la fin d’une phrase ;
- `compter_suivants` compte, pour chaque mot, les mots qui le suivent ;
- `predire` choisit le suivant le plus fréquent. En cas d’égalité, il prend le premier dans l’ordre des caractères.

Le texte est celui du chapitre 1.3, une quinzaine de lignes en français. Les suivants de « le » sont affichés dans l’ordre de la page : le plus fréquent d’abord, puis l’ordre des caractères.

In [ ]:
from mot_suivant.mots import charger_texte, compter_suivants, decouper_mots, generer, predire, premiere_reprise


def citer(mot):
    # Entre guillemets français. \u202f est l’espace fine insécable de la typographie française.
    return "«\u202f" + mot + "\u202f»"


texte = charger_texte()
mots_du_texte = decouper_mots(texte)
print(f"Le texte compte {len(texte)} caractères et {len(mots_du_texte)} mots.")
print("Ses douze premiers mots.", mots_du_texte[:12])

suivants = compter_suivants(mots_du_texte)
print(f"Les mots qui suivent {citer('le')}, avec leur nombre d’apparitions.")
# Trié par nombre décroissant (le signe moins), puis par ordre des caractères.
for suivant, nombre in sorted(suivants["le"].items(), key=lambda paire: (-paire[1], paire[0])):
    print(f"  {citer(suivant)}, {nombre} fois")
print(f"Le mot suivant le plus probable après {citer('le')} est {citer(predire(suivants, 'le'))}.")

## Étape 5 : une suite de 12 mots

`generer` part d’un mot, ajoute son suivant le plus probable, puis le suivant de ce suivant, et ainsi de suite. C’est déjà, en tout petit, ce que fait un modèle de langage : deviner le mot suivant, encore et encore.

Mais ici, le choix est toujours le même : après un mot donné, c’est toujours le même suivant. Dès qu’un mot revient, la suite répète donc les mêmes mots, sans fin. `premiere_reprise` trouve le premier mot qui revient.

La cellule écrit d’abord une suite de 12 mots, puis une de 30 : regarde si un mot revient. Change ensuite le mot de départ, puis relance la cellule. Un mot absent du texte n’a aucun suivant : la suite s’arrête aussitôt.

In [ ]:
depart = "le"

for longueur in (12, 30):
    suite = generer(suivants, depart, longueur)
    nom = "mot" if len(suite) == 1 else "mots"
    print(f"Une suite de {len(suite)} {nom}\u202f: {' '.join(suite)}")
    reprise = premiere_reprise(suite)
    if reprise is None:
        print("  Aucun mot ne revient dans cette suite.")
    else:
        print(f"  La suite revient sur {citer(suite[reprise])} au mot {reprise + 1}\u202f: à partir de là, elle tourne en rond.")

## Exercice : compter les mots

Écris la fonction `compter_mots(mots)`. Elle reçoit une liste de mots, et renvoie un `Counter` qui donne, pour chaque mot, son nombre d’apparitions dans la liste. Pour t’entraîner, écris la boucle toi-même, sans `Counter(mots)`.

La cellule essaie ta fonction sur les mots du texte : si le mot le plus fréquent et son nombre d’apparitions sont ceux que trouve `Counter`, le défi est réussi.

Indice : pars d’un `Counter()` vide, et pour chaque mot de la liste, ajoute 1 à son compte.

In [ ]:
from collections import Counter


def compter_mots(mots: list[str]) -> Counter:
    compte = Counter()
    # À toi : pour chaque mot de la liste, ajoute 1 à son compte.
    return compte


# Ne change rien en dessous : cette partie essaie ta fonction sur les mots du texte.
ton_compte = compter_mots(mots_du_texte)
attendu = Counter(mots_du_texte).most_common(1)
if ton_compte and Counter(ton_compte).most_common(1) == attendu:
    mot, nombre = attendu[0]
    print(f"Défi réussi\u202f: le mot le plus fréquent est {citer(mot)}, vu {nombre} fois.")
else:
    print("Pas encore. Ta fonction doit compter chaque mot de la liste.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

Une boucle sur la liste, et `+= 1` pour chaque mot. Un `Counter` vaut 0 pour un mot qu’il n’a pas encore vu : pas besoin de vérifier si le mot y est déjà.

Remplace la fonction de la cellule de l’exercice par :

```python
def compter_mots(mots: list[str]) -> Counter:
    compte = Counter()
    for mot in mots:
        compte[mot] += 1
    return compte
```

Le gagnant est le point : pour ce programme, la fin d’une phrase est un mot comme les autres.

</details>

## Dernière étape : le même résultat que la page ?

La page du site n’utilise pas Python : elle refait le même calcul en TypeScript, directement dans ton navigateur. Pour être sûr que les deux donnent exactement les mêmes mots, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes par le Python : pour plusieurs textes, les mots, leurs suivants, les prédictions et des suites de mots. Les tests de la page relisent ce fichier. La cellule suivante le relit aussi, et compare toutes les prédictions et toutes les suites du texte à ce que ton programme vient de calculer.

In [ ]:
import json

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
reference = attendus["textes"][0]   # le premier texte du fichier est celui du projet

# Chaque mot du texte une seule fois, dans l’ordre où il apparaît, avec sa prédiction.
predictions = [[mot, predire(suivants, mot)] for mot in dict.fromkeys(mots_du_texte)]
memes_predictions = reference["texte"] == texte and predictions == reference["predictions"]

# Chaque suite du fichier : depuis chaque mot du texte, de 1, 2, 12 et 30 mots.
suites_differentes = [s for s in reference["suites"] if generer(suivants, s["depart"], s["longueur"]) != s["suite"]]

print(f"{len(reference['predictions'])} prédictions et {len(reference['suites'])} suites comparées.")
if not memes_predictions:
    print("Les prédictions ne sont pas celles du fichier de référence.")
for s in suites_differentes[:3]:
    print(f"Suite différente depuis {citer(s['depart'])}, avec {s['longueur']} mots.")

identique = memes_predictions and not suites_differentes
print(f"Même résultat que la page du site\u202f: {'oui' if identique else 'non'}")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/0.1-python sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/0.1-python).
- La page du chapitre, pour retrouver l’atelier, les paris et le code expliqué : [chapitre 0.1](https://chipat-neko.github.io/forge-ia/partie-0/0-1/).
- Pour ne pas tourner en rond, un vrai modèle de langage ne prend pas toujours le mot le plus probable : il tire au sort, selon les probabilités. Le chapitre 0.2 montre comment.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.